# Fase 1 — Adquisición de datos y Análisis Exploratorio (EDA)

**Thesis:** Comparativa de algoritmos para la predicción de la demanda eléctrica  
**Author:** Antonio Navarro  

This notebook:
1. Downloads 5 years of ESIOS real demand (indicator 1293) at hourly resolution
2. Downloads hourly weather for 6 Spanish cities via Open-Meteo
3. Computes Spanish national + local holidays
4. Engineers the full feature set (cyclical, lags, rolling stats, HDD/CDD)
5. Performs EDA: STL decomposition, ADF stationarity test, ACF/PACF, Box-Cox, Pearson correlation
6. Splits 60/20/20 chronologically and saves CSVs to `data/`

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats

# Local utilities
sys.path.insert(0, os.path.abspath('.'))
import data_utils as du

# Reproducibility
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
sns.set_theme(style='whitegrid')

os.makedirs('data',  exist_ok=True)
os.makedirs('cache', exist_ok=True)
print('Environment ready.')

## 1 · Download ESIOS demand data (2020–2025)

In [ ]:
START_DATE = '2020-01-01'
END_DATE   = '2025-07-31'

df_demand = du.download_esios(
    indicator_id = du.ESIOS_REAL_DEMAND,
    start_date   = START_DATE,
    end_date     = END_DATE,
    api_key_file = 'esios_api_key.txt',
)

print(f"\nDemand data shape : {df_demand.shape}")
print(f"Date range        : {df_demand['datetime'].min()} → {df_demand['datetime'].max()}")
print(f"Missing values    : {df_demand['value'].isna().sum()}")
df_demand.head()

In [ ]:
# Quick sanity check: plot raw hourly demand
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(df_demand['datetime'], df_demand['value'], linewidth=0.4, color='steelblue')
ax.set_title('Spanish electricity demand — hourly (2020–2025)')
ax.set_ylabel('Demand (MW)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('data/fig_raw_demand.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean : {df_demand['value'].mean():.0f} MW")
print(f"Min  : {df_demand['value'].min():.0f} MW")
print(f"Max  : {df_demand['value'].max():.0f} MW")
print(f"Std  : {df_demand['value'].std():.0f} MW")

## 2 · Download Open-Meteo weather (6 Spanish cities)

In [ ]:
df_weather = du.download_openmeteo(
    start_date = START_DATE,
    end_date   = END_DATE,
)

print(f"\nWeather data shape : {df_weather.shape}")
print(f"Columns            : {list(df_weather.columns)}")
df_weather.head()

## 3 · Spanish holidays

In [ ]:
holidays = du.compute_spanish_holidays(start_year=2020, end_year=2025)
print(f"Total holiday entries (2020–2025): {len(holidays)}")
# Show a sample
for k, v in list(holidays.items())[:10]:
    print(f"  {k}: {v}")

## 4 · Feature engineering

In [ ]:
df = du.build_hourly_features(
    df_demand  = df_demand,
    df_weather = df_weather,
    df_holidays= holidays,
)

print(f"\nFull feature dataset: {df.shape}")
print(f"Columns ({len(df.columns)}):")
for c in df.columns:
    print(f"  {c}")

In [ ]:
# Check for remaining NaNs
nan_counts = df.isna().sum()
if nan_counts.any():
    print('Columns with NaN:')
    print(nan_counts[nan_counts > 0])
else:
    print('No NaN values — dataset is clean.')
df.describe().T

## 5 · EDA
### 5.1 STL Decomposition

In [ ]:
# STL on a representative year (2023) to keep computation manageable
ts_2023 = (
    df[df['datetime'].dt.year == 2023]
    .set_index('datetime')['demand_mw']
)

stl = STL(ts_2023, period=24, robust=True)  # period=24 → daily seasonality
res = stl.fit()

fig = res.plot()
fig.set_size_inches(14, 8)
fig.suptitle('STL Decomposition — hourly demand 2023 (period=24h)', y=1.01)
plt.tight_layout()
plt.savefig('data/fig_stl_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Trend  range : {res.trend.min():.0f} – {res.trend.max():.0f} MW")
print(f"Seasonal std : {res.seasonal.std():.0f} MW")
print(f"Residual std : {res.resid.std():.0f} MW")

In [ ]:
# Weekly seasonality — average demand by hour of day
df['hour'] = pd.to_datetime(df['datetime']).dt.hour
df['dow']  = pd.to_datetime(df['datetime']).dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hourly_avg = df.groupby('hour')['demand_mw'].mean()
axes[0].plot(hourly_avg.index, hourly_avg.values, marker='o', markersize=4)
axes[0].set_title('Average demand by hour of day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Mean demand (MW)')

day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_avg = df.groupby('dow')['demand_mw'].mean()
axes[1].bar(day_names, dow_avg.values, color='steelblue')
axes[1].set_title('Average demand by day of week')
axes[1].set_ylabel('Mean demand (MW)')

plt.tight_layout()
plt.savefig('data/fig_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Stationarity — Augmented Dickey-Fuller test

In [ ]:
series = df['demand_mw'].dropna()

result = adfuller(series, autolag='AIC')
print('Augmented Dickey-Fuller Test')
print(f'  Test statistic : {result[0]:.4f}')
print(f'  p-value        : {result[1]:.6f}')
print(f'  # lags used    : {result[2]}')
print(f'  # observations : {result[3]}')
for k, v in result[4].items():
    print(f'  Critical value ({k}): {v:.4f}')

if result[1] < 0.05:
    print('\n→ Series is STATIONARY (reject H0 at 5% level)')
else:
    print('\n→ Series is NON-STATIONARY (fail to reject H0)')

### 5.3 ACF and PACF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf( series, lags=72, ax=axes[0], alpha=0.05)
plot_pacf(series, lags=72, ax=axes[1], alpha=0.05, method='ywm')
axes[0].set_title('ACF — hourly demand (lags 0–72h)')
axes[1].set_title('PACF — hourly demand (lags 0–72h)')
for ax in axes:
    ax.set_xlabel('Lag (hours)')
plt.tight_layout()
plt.savefig('data/fig_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Box-Cox test (variance stabilisation)

In [ ]:
transformed, lam = stats.boxcox(series.values)
print(f'Optimal Box-Cox lambda: {lam:.4f}')
print('(λ≈1 → no transformation needed; λ≈0 → log transform)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(series.values,  bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Original demand distribution')
axes[1].hist(transformed, bins=60, color='darkorange', edgecolor='white')
axes[1].set_title(f'Box-Cox transformed (λ={lam:.3f})')
plt.tight_layout()
plt.savefig('data/fig_boxcox.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Pearson correlation — demand vs exogenous variables

In [ ]:
# Select a subset of features for correlation analysis
exog_cols = (
    [f"{c}_{v}" for c in du.CITIES for v in du.WEATHER_VARS]
    + ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
       'is_weekday', 'is_saturday', 'is_sunday', 'is_national_holiday',
       'demand_lag_24h', 'demand_lag_168h',
       'demand_roll24_mean', 'demand_roll168_mean', 'hdd', 'cdd']
)
exog_cols = [c for c in exog_cols if c in df.columns]

corr = df[['demand_mw'] + exog_cols].corr()['demand_mw'].drop('demand_mw').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, max(5, len(corr) * 0.25)))
colors = ['#d73027' if v > 0 else '#4575b4' for v in corr.values]
corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson correlation with hourly demand')
ax.set_xlabel('Correlation coefficient')
plt.tight_layout()
plt.savefig('data/fig_pearson_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features by |correlation|:')
print(corr.head(10).to_string())

## 6 · Chronological split (60 / 20 / 20) and save to `data/`

In [ ]:
df_train, df_val, df_test = du.chronological_split(df, train_frac=0.60, val_frac=0.20)

print(f"\nTrain : {df_train['datetime'].min().date()} → {df_train['datetime'].max().date()}")
print(f"Val   : {df_val['datetime'].min().date()} → {df_val['datetime'].max().date()}")
print(f"Test  : {df_test['datetime'].min().date()} → {df_test['datetime'].max().date()}")

In [ ]:
# Visualise the split
fig, ax = plt.subplots(figsize=(15, 3))
ax.plot(df_train['datetime'], df_train['demand_mw'], color='#1f77b4', linewidth=0.4, label='Train')
ax.plot(df_val['datetime'],   df_val['demand_mw'],   color='#ff7f0e', linewidth=0.4, label='Val')
ax.plot(df_test['datetime'],  df_test['demand_mw'],  color='#2ca02c', linewidth=0.4, label='Test')
ax.set_title('Demand split: train / validation / test')
ax.set_ylabel('Demand (MW)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.savefig('data/fig_data_split.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save full feature dataset and splits
df.to_csv('data/hourly_features_full.csv', index=False)
df_train.to_csv('data/features_train.csv', index=False)
df_val.to_csv(  'data/features_val.csv',   index=False)
df_test.to_csv( 'data/features_test.csv',  index=False)

print('Saved:')
for name, d in [('full', df), ('train', df_train), ('val', df_val), ('test', df_test)]:
    path = f'data/hourly_features_full.csv' if name == 'full' else f'data/features_{name}.csv'
    size_mb = os.path.getsize(path) / 1024**2
    print(f"  data/{'hourly_features_full.csv' if name == 'full' else f'features_{name}.csv'} — {len(d):,} rows — {size_mb:.1f} MB")

## Summary

| Dataset  | Rows  | Date range |
|----------|-------|------------|
| Train    | ~29k  | 2020-01 → 2023-03 |
| Val      | ~9.7k | 2023-03 → 2024-03 |
| Test     | ~9.7k | 2024-03 → 2025-07 |

**Key EDA findings** (fill in after running):
- ADF p-value: …  (stationary / non-stationary)
- Box-Cox λ: … (transformation needed?)
- Top correlated features: demand_lag_24h, demand_roll24_mean, temperature, hour_cos…
- Dominant seasonalities: 24h (daily), 168h (weekly), annual

Next step → `fase2_1_sarimax.ipynb`